In [1]:
# ============================================================
# MISSION 3 — BRIQUE 1 : Nettoyage du catalogue produits
# Surenthini SIVAKUMAR — Nesri Discount
# ============================================================

import pandas as pd
import re
import os

# ============================================================
# 1. CHARGEMENT DES 5 FICHIERS CSV
# ============================================================

dossier_brut = "/Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/01_Donnees_brutes/"

dfs = []
for i in range(1, 6):
    chemin = os.path.join(dossier_brut, f"products_export_{i}.csv")
    df = pd.read_csv(chemin, low_memory=False)
    print(f"Fichier {i} chargé : {len(df)} lignes")
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal brut : {len(df_all)} lignes")

# ============================================================
# 2. NETTOYAGE
# ============================================================

# Étape 1 — Garder uniquement les produits actifs
df = df_all[df_all['Status'] == 'active'].copy()
print(f"Après filtre 'active' : {len(df)} lignes")

# Étape 2 — Garder uniquement les produits publiés
df = df[df['Published'] == True].copy()
print(f"Après filtre 'Published' : {len(df)} lignes")

# Étape 3 — Garder uniquement les produits en stock
df = df[pd.to_numeric(df['Variant Inventory Qty'], errors='coerce').fillna(0) > 0].copy()
print(f"Après filtre stock > 0 : {len(df)} lignes")

# Étape 4 — Supprimer les doublons de variantes (1 ligne par produit)
df = df.drop_duplicates(subset='Handle', keep='first').copy()
print(f"Après déduplication : {len(df)} lignes")

# Étape 5 — Nettoyer et préparer les deux prix
df['Prix_vente'] = pd.to_numeric(df['Variant Price'], errors='coerce')
df['Prix_barre'] = pd.to_numeric(df['Variant Compare At Price'], errors='coerce')

# Supprimer les produits sans prix de vente
df = df[df['Prix_vente'] > 0].copy()
print(f"Après filtre prix > 0 : {len(df)} lignes")

# Étape 6 — Calculer l'économie et le % de remise
df['Economie'] = (df['Prix_barre'] - df['Prix_vente']).round(2)
df['Remise_pct'] = ((df['Economie'] / df['Prix_barre']) * 100).round(0)

# Étape 7 — Supprimer les remises aberrantes (erreurs de saisie Shopify)
nb_avant = len(df)
df = df[~(df['Remise_pct'] > 70)].copy()
print(f"Remises aberrantes supprimées : {nb_avant - len(df)} produits")

# Étape 8 — Nettoyer les descriptions HTML
def clean_html(text):
    if pd.isna(text):
        return ""
    text = re.sub(r'<[^>]+>', ' ', str(text))   # supprimer balises HTML
    text = re.sub(r'&[a-z]+;', ' ', text)        # supprimer &nbsp; etc.
    text = re.sub(r'\s+', ' ', text).strip()     # supprimer espaces en trop
    return text[:600]                             # limiter à 600 caractères

df['Description'] = df['Body (HTML)'].apply(clean_html)

# ============================================================
# 3. GARDER UNIQUEMENT LES COLONNES UTILES
# ============================================================

df_final = df[[
    'Handle',
    'Title',
    'Vendor',
    'Prix_vente',
    'Prix_barre',
    'Economie',
    'Remise_pct',
    'Description',
    'Image Src',
    'Marques (product.metafields.custom.marques)',
    'Type',
    'Product Category'
]].copy()

# Renommer les colonnes en français
df_final.columns = [
    'Handle',
    'Titre',
    'Vendeur',
    'Prix_vente',
    'Prix_barre',
    'Economie',
    'Remise_pct',
    'Description',
    'Image',
    'Marque',
    'Type',
    'Categorie'
]

# Réinitialiser l'index pour la rotation
df_final = df_final.reset_index(drop=True)

# ============================================================
# 4. RÉSUMÉ FINAL
# ============================================================

print(f"\n{'='*50}")
print(f"✅ CATALOGUE FINAL : {len(df_final)} produits propres")
print(f"   → {df_final['Prix_barre'].notna().sum()} produits EN PROMO (avec prix barré)")
print(f"   → {df_final['Prix_barre'].isna().sum()} produits à prix fixe")
print(f"   → Remise moyenne : {df_final['Remise_pct'].mean():.0f}%")
print(f"{'='*50}")

# ============================================================
# RÉORGANISATION : produits EN PROMO en premier
# ============================================================

df_promo     = df_final[df_final['Prix_barre'].notna()].copy()
df_prix_fixe = df_final[df_final['Prix_barre'].isna()].copy()

# Trier les promos par économie décroissante (les meilleures deals en premier)
df_promo = df_promo.sort_values('Economie', ascending=False)

# Fusionner : promos d'abord, puis prix fixes
df_final = pd.concat([df_promo, df_prix_fixe], ignore_index=True)

print(f"\n📊 Ordre de rotation :")
print(f"   → Produits 0 à {len(df_promo)-1} : EN PROMO ({len(df_promo)} produits) 🔥")
print(f"   → Produits {len(df_promo)} à {len(df_final)-1} : Prix fixe ({len(df_prix_fixe)} produits)")

# ============================================================
# 5. SAUVEGARDE
# ============================================================

dossier_propre = "/Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/03_Donnees_propres/"

chemin_sortie = os.path.join(dossier_propre, "catalogue_produits_propre.csv")
df_final.to_csv(chemin_sortie, index=False, encoding='utf-8-sig')

print(f"\n💾 Fichier sauvegardé ici :")
print(f"   {chemin_sortie}")

Fichier 1 chargé : 9133 lignes
Fichier 2 chargé : 4445 lignes
Fichier 3 chargé : 3418 lignes
Fichier 4 chargé : 4068 lignes
Fichier 5 chargé : 2711 lignes

Total brut : 23775 lignes
Après filtre 'active' : 4690 lignes
Après filtre 'Published' : 4674 lignes
Après filtre stock > 0 : 3779 lignes
Après déduplication : 3779 lignes
Après filtre prix > 0 : 3778 lignes
Remises aberrantes supprimées : 2 produits

✅ CATALOGUE FINAL : 3776 produits propres
   → 1835 produits EN PROMO (avec prix barré)
   → 1941 produits à prix fixe
   → Remise moyenne : 20%

📊 Ordre de rotation :
   → Produits 0 à 1834 : EN PROMO (1835 produits) 🔥
   → Produits 1835 à 3775 : Prix fixe (1941 produits)

💾 Fichier sauvegardé ici :
   /Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/03_Donnees_propres/catalogue_produits_propre.csv


In [2]:
# ============================================================
# MISSION 3 — BRIQUE 2 : Sélection du produit du jour
# Surenthini SIVAKUMAR — Nesri Discount
# ============================================================

import pandas as pd
import os

# ============================================================
# CHEMINS
# ============================================================

dossier_propre = "/Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/03_Donnees_propres/"

chemin_catalogue = os.path.join(dossier_propre, "catalogue_produits_propre.csv")
chemin_compteur  = os.path.join(dossier_propre, "compteur_rotation.txt")

# ============================================================
# 1. CHARGER LE CATALOGUE
# ============================================================

df = pd.read_csv(chemin_catalogue)
nb_produits = len(df)
print(f"Catalogue chargé : {nb_produits} produits")

# ============================================================
# 2. LIRE LE COMPTEUR (quel produit a été envoyé hier ?)
# ============================================================

# Si le fichier compteur n'existe pas encore → on commence à 0
if not os.path.exists(chemin_compteur):
    index_hier = -1
    print("Premier démarrage — aucun compteur existant")
else:
    with open(chemin_compteur, 'r') as f:
        index_hier = int(f.read().strip())
    print(f"Dernier produit envoyé : index {index_hier}")

# ============================================================
# 3. CALCULER L'INDEX DU PRODUIT DU JOUR
# ============================================================

# On prend le suivant, et on recommence à 0 quand on arrive à la fin
index_aujourdhui = (index_hier + 1) % nb_produits
print(f"Produit du jour : index {index_aujourdhui}")

# ============================================================
# 4. RÉCUPÉRER LES INFOS DU PRODUIT
# ============================================================

# APRÈS — on règle le nan sur la marque
produit = df.iloc[index_aujourdhui].copy()
if pd.isna(produit['Marque']) or str(produit['Marque']).strip() == 'nan':
    produit['Marque'] = produit['Vendeur']
    
print(f"\n{'='*50}")
print(f"📦 PRODUIT DU JOUR")
print(f"{'='*50}")
print(f"Titre      : {produit['Titre']}")
print(f"Marque     : {produit['Marque']}")
print(f"Vendeur    : {produit['Vendeur']}")

if pd.notna(produit['Prix_barre']):
    print(f"Prix       : ~~{produit['Prix_barre']}€~~ → {produit['Prix_vente']}€")
    print(f"Économie   : -{int(produit['Remise_pct'])}% soit {produit['Economie']}€ d'économie")
else:
    print(f"Prix       : {produit['Prix_vente']}€")

print(f"Description: {produit['Description'][:150]}...")
print(f"Image      : {produit['Image']}")
print(f"{'='*50}")

# ============================================================
# 5. METTRE À JOUR LE COMPTEUR POUR DEMAIN
# ============================================================

with open(chemin_compteur, 'w') as f:
    f.write(str(index_aujourdhui))

print(f"\n✅ Compteur mis à jour → demain ce sera le produit {index_aujourdhui + 1}")

Catalogue chargé : 3776 produits
Dernier produit envoyé : index 41
Produit du jour : index 42

📦 PRODUIT DU JOUR
Titre      : N 90, Machine à café tout automatique, Inox NEFF CL4TT11N0
Marque     : NEFF
Vendeur    : NEFF
Prix       : ~~2702.09€~~ → 2079.0€
Économie   : -23% soit 623.09€ d'économie
Description: N 90, Machine à café tout automatique, Inox NEFF CL4TT11N0 Goût aromaDouble Shot – moins d'amertume pour les grandes boissons grâce à deux procédés de...
Image      : https://cdn.shopify.com/s/files/1/0982/1428/1557/files/1_7cda7bd5-b816-45d4-b0ff-30585f6a9b79.png?v=1769708068

✅ Compteur mis à jour → demain ce sera le produit 43


In [3]:
# ============================================================
# MISSION 3 — BRIQUE 3 : Génération du post Instagram via IA
# Surenthini SIVAKUMAR — Nesri Discount
# ============================================================
import pandas as pd
import os
from groq import Groq
from dotenv import load_dotenv

# ============================================================
# 1. CHARGER LA CLÉ API
# ============================================================
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# ============================================================
# 2. CHARGER LE CATALOGUE ET SÉLECTIONNER LE PRODUIT DU JOUR
# ============================================================
dossier_propre = "/Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/03_Donnees_propres/"

df = pd.read_csv(os.path.join(dossier_propre, "catalogue_produits_propre.csv"))
chemin_compteur = os.path.join(dossier_propre, "compteur_rotation.txt")

# Lire le compteur
if not os.path.exists(chemin_compteur):
    index_hier = -1
else:
    with open(chemin_compteur, 'r') as f:
        index_hier = int(f.read().strip())

index_aujourdhui = (index_hier + 1) % len(df)

# Récupérer le produit
produit = df.iloc[index_aujourdhui].copy()
if pd.isna(produit['Marque']) or str(produit['Marque']).strip() == 'nan':
    produit['Marque'] = produit['Vendeur']

# Formater les prix sans décimales inutiles  # 👈 AJOUTÉ
def format_prix(prix):                        # 👈 AJOUTÉ
    if pd.isna(prix):                         # 👈 AJOUTÉ
        return ""                             # 👈 AJOUTÉ
    return f"{int(prix):,}€".replace(",", " ")# 👈 AJOUTÉ
                                              
produit['Prix_vente_affiche'] = format_prix(produit['Prix_vente'])  # 👈 AJOUTÉ
produit['Prix_barre_affiche'] = format_prix(produit['Prix_barre'])  # 👈 AJOUTÉ
produit['Economie_affiche']   = format_prix(produit['Economie'])    # 👈 AJOUTÉ

# ============================================================
# 3. CONSTRUIRE LE PROMPT SELON SI PROMO OU PAS
# ============================================================
if pd.notna(produit['Prix_barre']):
    info_prix = (
        f"Prix original : {produit['Prix_barre_affiche']}\n"   # 👈 MODIFIÉ
        f"Prix actuel : {produit['Prix_vente_affiche']}\n"     # 👈 MODIFIÉ
        f"Économie : {produit['Economie_affiche']} soit -{int(produit['Remise_pct'])}% de remise"  # 👈 MODIFIÉ
    )
else:
    info_prix = f"Prix : {produit['Prix_vente_affiche']}"      # 👈 MODIFIÉ

prompt = f"""Tu es un expert en marketing digital pour une boutique en ligne française d'électroménager discount appelée Nesri Discount.

Génère un post Instagram accrocheur pour le produit suivant :

Nom du produit : {produit['Titre']}
Marque : {produit['Marque']}
{info_prix}
Description : {produit['Description'][:400]}

Règles importantes :
- Commence par une phrase d'accroche percutante avec des emojis
- Mets en valeur le prix et l'économie si le produit est en promo
- Les hashtags doivent être sans accents, pertinents au produit et à la marque, maximum 8, jamais #Achetez ou #Discount seul
- Le ton doit être enthousiaste et dynamique comme un vrai community manager
- Le post doit faire entre 6 et 10 lignes, avec des sauts de ligne entre chaque idée pour être agréable à lire sur Instagram
- Écris en français
- Termine toujours par : 🛒 Disponible sur NesriDiscount.fr
"""

# ============================================================
# 4. ENVOYER À L'IA ET RÉCUPÉRER LE POST
# ============================================================
print(f"Génération du post pour : {produit['Titre'][:60]}...")
print("="*50)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=500,
    temperature=0.8
)

post_genere = response.choices[0].message.content

print("\n📱 POST INSTAGRAM GÉNÉRÉ :")
print("="*50)
print(post_genere)
print("="*50)

# ============================================================
# 5. METTRE À JOUR LE COMPTEUR
# ============================================================
with open(chemin_compteur, 'w') as f:
    f.write(str(index_aujourdhui))

print(f"\n✅ Compteur mis à jour → demain ce sera le produit {index_aujourdhui + 1}")

Génération du post pour : Série 8, Machine à café tout automatique, Noir BOSCH CTL7181...

📱 POST INSTAGRAM GÉNÉRÉ :
🚨 Réveillez vos sens avec la meilleure machine à café de l'année ! ☕️
Notre Série 8, Machine à café tout automatique, Noir BOSCH CTL7181B0 est maintenant à 2 079€, soit -23% de remise par rapport à son prix original de 2 702€, vous économisez 623€ ! 💸
Vous pouvez personnaliser jusqu'à 30 boissons et profiter de 32 recettes de café différentes avec cette incroyable machine.
Elle dispose d'un écran couleur TFT tactile de 6,8" pour un confort d'utilisation optimal.
La marque Bosch garantit une qualité et une sécurité exceptionnelles.
Cette offre exceptionnelle ne durera pas éternellement, profitez-en vite ! #Bosch #MachineACafe #Cafe #Electromenager #NesriDiscount #Promo #Economie #CafeLover
🛒 Disponible sur NesriDiscount.fr

✅ Compteur mis à jour → demain ce sera le produit 44


In [ ]:
# ============================================================
# MISSION 3 — FUSION COMPLET — CATALOGUE MULTI-PRODUITS (4 PAR JOUR)
# Surenthini SIVAKUMAR — Nesri Discount
# ============================================================
import os
import smtplib
from email import encoders
from email.mime.base import MIMEBase
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from io import BytesIO
import requests
import pandas as pd
from PIL import Image, ImageDraw, ImageFilter, ImageFont
from dotenv import load_dotenv
from groq import Groq

# ============================================================
# 1. INITIALISATION ET CONFIGURATION
# ============================================================
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

expediteur = "raadha.2910@gmail.com"
mot_de_passe = "kzfd msgr ykwg duse"
destinataire = "shivadheepi@gmail.com"

CONFIG_IMAGES = {
    "INSTA": {"size": (1080, 1350)},
    "TIKTOK": {"size": (1080, 1920)},
}

COLOR_BG_TOP = (12, 22, 54, 255)       # Bleu nuit moderne pro
COLOR_BG_BOTTOM = (20, 32, 68, 255)
COLOR_TEXT_MAIN = (255, 255, 255, 255) # Blanc pur
COLOR_TEXT_MUTED = (170, 185, 220, 255)# Bleu épuré
COLOR_ACCENT = (0, 245, 156, 255)      # Vert néon
COLOR_RED = (227, 6, 19, 255)          # Rouge Nesri officiel

# ============================================================
# 2. CHARGEMENT ET SÉLECTION DES 4 PRODUITS DU JOUR
# ============================================================
dossier_propre = "/Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/03_Donnees_propres/"
df = pd.read_csv(os.path.join(dossier_propre, "catalogue_produits_propre.csv"))

mots_cles_ete = ["frigo", "réfrigérateur", "refrigerateur", "ventilateur", "climatiseur", 
                 "climatisation", "rafraîchisseur", "rafraichisseur", "congélateur", 
                 "congelateur", "glacière", "glaciere", "barbecue", "plancha"]

masque_ete = df['Titre'].str.lower().str.contains('|'.join(mots_cles_ete)) | \
             df['Description'].str.lower().str.contains('|'.join(mots_cles_ete)) | \
             df['Categorie'].str.lower().str.contains('|'.join(mots_cles_ete))

df_ete = df[masque_ete].copy()

df_promos_ete = df_ete[df_ete['Remise_pct'] > 0].copy()
df_promos_ete = df_promos_ete.sort_values(by=['Remise_pct', 'Economie'], ascending=False)

if len(df_promos_ete) < 4:
    df_promos_ete = df_ete.sort_values(by=['Prix_vente'], ascending=True)

chemin_compteur = os.path.join(dossier_propre, "compteur_rotation_ete.txt")

if not os.path.exists(chemin_compteur):
    index_hier = -1
else:
    with open(chemin_compteur, 'r') as f:
        index_hier = int(f.read().strip())

# Sélection de 4 produits consécutifs pour assurer la rotation quotidienne
indices_aujourdhui = [(index_hier + i + 1) % len(df_promos_ete) for i in range(4)]
liste_produits = [df_promos_ete.iloc[idx].copy() for idx in indices_aujourdhui]

# Nettoyage des marques
for prod in liste_produits:
    if pd.isna(prod['Marque']) or str(prod['Marque']).strip() == 'nan':
        prod['Marque'] = prod['Vendeur']

# Formatage des affichages
def format_prix(prix):
    if pd.isna(prix):
        return ""
    return f"{int(prix):,}€".replace(",", " ")

for prod in liste_produits:
    prod['Prix_vente_affiche'] = format_prix(prod['Prix_vente'])
    prod['Prix_barre_affiche'] = format_prix(prod['Prix_barre'])
    prod['Economie_affiche']   = format_prix(prod['Economie'])

# 🛡️ VALIDATION DES PRIX POUR LES 4 PRODUITS
print("-" * 60)
print("🛡️ SÉCURITÉ CATALOGUE — VÉRIFICATION DES 4 PRODUITS DU JOUR")
print("-" * 60)
for i, prod in enumerate(liste_produits, 1):
    print(f"📦 [{i}/4] {prod['Titre'][:50]}... | 💰 {prod['Prix_vente_affiche']} (Barre: {prod['Prix_barre_affiche']})")
print("-" * 60)

confirmation = input("👉 Les prix de vente sont-ils corrects et à jour ? (OUI/NON) : ").strip().upper()
if confirmation in ["NON", "N"]:
    print("\n⚠️ Veuillez relancer le script après avoir mis à jour votre catalogue source ou modifiez directement le CSV.")
    exit()

# ============================================================
# 3. FONCTIONS POUR LE RENDU VISUEL ET LE NLP
# ============================================================
def load_font(size, bold=False):
    candidates = [
        "/Library/Fonts/Arial Bold.ttf" if bold else "/Library/Fonts/Arial.ttf",
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
    ]
    for path in candidates:
        if os.path.exists(path):
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()

def text_width(draw, text, font):
    bbox = draw.textbbox((0, 0), text, font=font)
    return bbox[2] - bbox[0]

def fit_font(draw, text, max_width, start=40, min_size=26, bold=True):
    for size in range(start, min_size - 1, -2):
        font = load_font(size, bold=bold)
        if text_width(draw, text, font) <= max_width:
            return font
    return load_font(min_size, bold=bold)

def wrap_text(draw, text, font, max_width, max_lines=2):
    words = text.split()
    lines = []
    current = ""
    for w in words:
        test = w if not current else current + " " + w
        if text_width(draw, test, font) <= max_width:
            current = test
        else:
            if current:
                lines.append(current)
            current = w
    if current:
        lines.append(current)
    return "\n".join(lines[:max_lines])

def safe_int(v, default=0):
    try:
        if v is None or pd.isna(v):
            return default
        return int(float(v))
    except:
        return default

def generer_marketing_post(produit_data):
    categorie_produit = str(produit_data.get('Categorie', '')).lower()
    titre_produit = str(produit_data['Titre']).lower()
    consigne_style = "Adopte un ton dynamique et estival."

    if any(x in titre_produit or x in categorie_produit for x in ["froid", "refrigerateur", "frigo", "congelateur", "congélateur"]):
        consigne_style = "Mets l'accent sur la fraîcheur indispensable en plein été ! Parle de conserver au frais."
    elif any(x in titre_produit or x in categorie_produit for x in ["ventilateur", "climatiseur", "rafraichisseur", "rafraîchisseur"]):
        consigne_style = "Insiste sur la canicule et la chaleur. C'est la solution pour respirer chez soi."
    elif "barbecue" in categorie_produit or "camping" in categorie_produit:
        consigne_style = "Mets l'accent sur les soirées d'été en extérieur et la convivialité."

    info_prix = f"Prix : {produit_data['Prix_vente_affiche']} (Au lieu de {produit_data['Prix_barre_affiche']})" if pd.notna(produit_data['Prix_barre']) else f"Prix : {produit_data['Prix_vente_affiche']}"

    prompt = f"""Génère un post Instagram court (4-6 lignes) pour : {produit_data['Titre']} ({produit_data['Marque']}). {info_prix}. Style : {consigne_style}. Utilise des emojis d'été, max 5 hashtags à la fin."""
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200,
            temperature=0.7
        )
        return response.choices[0].message.content
    except:
        return f"🔥 Offre Exceptionnelle sur le {produit_data['Titre']} ! Idéal pour cet été. Disponible sur NesriDiscount.fr"

# ============================================================
# 4. MOTEUR GRAPHIQUE AMÉLIORÉ AVEC VÉRIFICATION ANTI-COLLISION
# ============================================================
def generate_creative_in_memory(format_type, produit_data):
    W, H = CONFIG_IMAGES[format_type]["size"]
    canvas = Image.new("RGBA", (W, H))
    draw = ImageDraw.Draw(canvas)

    # Fond dégradé
    for y in range(H):
        t = y / H
        r = int(COLOR_BG_TOP[0] + (COLOR_BG_BOTTOM[0] - COLOR_BG_TOP[0]) * t)
        g = int(COLOR_BG_TOP[1] + (COLOR_BG_BOTTOM[1] - COLOR_BG_TOP[1]) * t)
        b = int(COLOR_BG_TOP[2] + (COLOR_BG_BOTTOM[2] - COLOR_BG_TOP[2]) * t)
        draw.line([(0, y), (W, y)], fill=(r, g, b, 255))

    # Fine bordure
    draw.rectangle([40, 40, W - 40, H - 40], outline=(255, 255, 255, 15), width=2)

    # Bandeau Flash
    bandeau = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    bd = ImageDraw.Draw(bandeau)
    bd.polygon([(W - 350, 0), (W, 350), (W, 230), (W - 230, 0)], fill=COLOR_RED)
    canvas = Image.alpha_composite(canvas, bandeau)
    draw = ImageDraw.Draw(canvas)
    
    f_flash = load_font(24, bold=True)
    draw.text((W - 195, 75), "OFFRE", font=f_flash, fill=(255, 255, 255, 255))
    draw.text((W - 185, 110), "FLASH", font=f_flash, fill=(255, 255, 255, 255))

    # Logo Nesri
    logo_path = "/Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/logo_nesri.png"
    if os.path.exists(logo_path):
        try:
            logo = Image.open(logo_path).convert("RGBA")
            logo_w = 360  
            logo_h = int(logo.height * (logo_w / logo.width))
            logo = logo.resize((logo_w, logo_h))
            canvas.alpha_composite(logo, (60, 60))
        except:
            pass

    # Image Produit Centrale positionnée plus haut pour donner de l'espace vertical
    py_prod = int(H * 0.20) if format_type == "TIKTOK" else int(H * 0.16)
    h_prod_box = int(W * 0.55)

    image_url = produit_data.get("Image")
    if pd.notna(image_url) and str(image_url).strip() != "":
        try:
            res = requests.get(image_url, timeout=10)
            img_prod = Image.open(BytesIO(res.content)).convert("RGBA")
            img_prod.thumbnail((h_prod_box, h_prod_box))
            px = (W - img_prod.width) // 2
            canvas.alpha_composite(img_prod, (px, py_prod))
        except:
            pass

    # Zone de texte stabilisée (Ligne de départ descendue pour laisser l'image tranquille)
    text_y_start = int(H * 0.64) if format_type == "TIKTOK" else int(H * 0.58)
    
    marque_txt = str(produit_data.get("Marque", "EXCLUSIVITÉ")).upper()
    draw.text((60, text_y_start), marque_txt, font=load_font(24, bold=True), fill=COLOR_TEXT_MUTED)

    titre_full = str(produit_data.get("Titre", "Produit"))
    title_font = fit_font(draw, titre_full, W - 120, start=38, min_size=26, bold=True)
    title_wrapped = wrap_text(draw, titre_full, title_font, W - 120)
    
    draw.multiline_text((60, text_y_start + 40), title_wrapped, font=title_font, fill=COLOR_TEXT_MAIN, spacing=6)
    
    # 🛡️ POSITIONNEMENT DYNAMIQUE ANTI-COLLISION DU PRIX
    bbox_titre = draw.multiline_textbbox((60, text_y_start + 40), title_wrapped, font=title_font, spacing=6)
    hauteur_titre = bbox_titre[3] - bbox_titre[1]
    prix_y = text_y_start + 40 + hauteur_titre + 40  # Marge claire de 40px

    prix_vente = safe_int(produit_data.get("Prix_vente"), 0)
    prix_barre = safe_int(produit_data.get("Prix_barre"), 0)
    remise = safe_int(produit_data.get("Remise_pct"), 0)

    txt_pv = f"{prix_vente} €"
    draw.text((60, prix_y), txt_pv, font=load_font(85, bold=True), fill=COLOR_TEXT_MAIN)

    if prix_barre > 0 and remise > 0:
        largeur_pv = text_width(draw, txt_pv, load_font(85, bold=True))
        next_x = 60 + largeur_pv + 30
        
        txt_pb = f"{prix_barre} €"
        draw.text((next_x, prix_y + 35), txt_pb, font=load_font(36), fill=COLOR_TEXT_MUTED)
        
        largeur_pb = text_width(draw, txt_pb, load_font(36))
        draw.line([(next_x, prix_y + 53), (next_x + largeur_pb, prix_y + 53)], fill=COLOR_RED, width=3)

        draw.rounded_rectangle([W - 240, prix_y + 10, W - 60, prix_y + 75], radius=14, fill=COLOR_ACCENT)
        txt_remise = f"-{remise}%"
        tw_rem = text_width(draw, txt_remise, load_font(32, bold=True))
        draw.text((W - 150 - (tw_rem // 2), prix_y + 22), txt_remise, font=load_font(32, bold=True), fill=(12, 22, 54, 255))

    # Bouton d'action stable tout en bas
    draw.rounded_rectangle([60, H - 140, W - 60, H - 60], radius=22, fill=COLOR_RED)
    cta_txt = "PROFITER DE L'OFFRE SUR LE SITE  ➔"
    tw_cta = text_width(draw, cta_txt, load_font(28, bold=True))
    draw.text(((W - tw_cta) // 2, H - 108), cta_txt, font=load_font(28, bold=True), fill=(255, 255, 255, 255))

    img_byte_arr = BytesIO()
    canvas.convert("RGB").save(img_byte_arr, format="PNG", optimize=True)
    img_byte_arr.seek(0)
    return img_byte_arr.read()

# ============================================================
# 5. CONSTRUCTION DE LA NEWSLETTER HTML MULTI-PRODUITS
# ============================================================
print("🤖 IA : Rédaction des textes marketing pour les 4 produits...")
blocs_html_produits = ""

message = MIMEMultipart("mixed")
message["From"] = expediteur
message["To"] = destinataire
message["Subject"] = "☀️ Les 4 Offres Flash Canicule du Jour - Nesri Discount"

for i, prod in enumerate(liste_produits, 1):
    desc_marketing = generer_marketing_post(prod)
    
    badge_promo = f'<span style="background:#E30613;color:#fff;font-size:12px;font-weight:bold;padding:3px 8px;border-radius:4px;margin-left:8px;">-{int(prod["Remise_pct"])}%</span>' if pd.notna(prod['Prix_barre']) else ''
    prix_barre_html = f'<span style="color:#999;text-decoration:line-through;margin-right:8px;font-size:14px;">{prod["Prix_barre_affiche"]}</span>' if pd.notna(prod['Prix_barre']) else ''

    blocs_html_produits += f"""
    <div style="border: 1px solid #eee; border-radius: 8px; padding: 16px; margin-bottom: 20px; background: #fafafa;">
      <table width="100%" cellspacing="0" cellpadding="0">
        <tr>
          <td width="30%" align="center">
            <img src="{prod['Image']}" style="max-width:120px; max-height:120px; object-fit:contain; display:block;" />
          </td>
          <td width="70%" style="padding-left:15px; font-family:Arial,sans-serif;">
            <h3 style="margin:0 0 6px 0; color:#1a1a1a; font-size:16px;">{prod['Titre']}</h3>
            <p style="margin:0 0 8px 0; color:#E30613; font-weight:bold; font-size:18px;">
              {prix_barre_html}{prod['Prix_vente_affiche']} {badge_promo}
            </p>
            <p style="margin:0; font-size:13px; color:#444; line-height:1.5; font-style:italic;">{desc_marketing}</p>
          </td>
        </tr>
      </table>
    </div>
    """
    
    # Génération et fixation des pièces jointes des images pour ce produit
    print(f"🎨 Génération des rendus HD pour le produit {i}/4...")
    img_insta = generate_creative_in_memory("INSTA", prod)
    img_tiktok = generate_creative_in_memory("TIKTOK", prod)
    
    for format_name, data in [("instagram", img_insta), ("tiktok", img_tiktok)]:
        part = MIMEBase("application", "octet-stream")
        part.set_payload(data)
        encoders.encode_base64(part)
        part.add_header("Content-Disposition", f'attachment; filename="produit_{i}_{format_name}.png"')
        message.attach(part)

html_global = f"""
<!DOCTYPE html>
<html>
<body style="margin:0;padding:0;background:#f4f4f4;font-family:Arial,sans-serif;">
<div style="max-width:600px;margin:20px auto;background:#ffffff;border-radius:12px;overflow:hidden;box-shadow:0 4px 10px rgba(0,0,0,0.1);">
  <div style="background:#0c1636;padding:24px;text-align:center;">
    <h1 style="color:#fff;font-size:26px;margin:0;letter-spacing:1px;">NESRI DISCOUNT</h1>
    <p style="color:#00f59c;font-size:14px;margin:5px 0 0;font-weight:bold;">🔥 SÉLECTION SPÉCIALE CANICULE — 4 PRODUITS PAR JOUR</p>
  </div>
  <div style="padding:20px 25px;">
    <p style="font-size:15px; color:#333; line-height:1.6;">Bonjour équipe communication,</p>
    <p style="font-size:14px; color:#666; margin-bottom:20px;">Voici votre sélection automatisée de 4 produits indispensables pour aujourd'hui avec leurs kits médias prêts à publier attachés à cet e-mail.</p>
    
    {blocs_html_produits}
    
    <div style="text-align:center; margin-top:25px;">
        <a href="https://nesridiscount.fr" style="background:#E30613; color:#fff; text-decoration:none; padding:12px 24px; border-radius:6px; font-weight:bold; display:inline-block;">Accéder au site internet officiel</a>
    </div>
  </div>
</div>
</body>
</html>
"""

msg_alternative = MIMEMultipart("alternative")
msg_alternative.attach(MIMEText(html_global, "html"))
message.attach(msg_alternative)

# ============================================================
# 6. EXPÉDITION DU MAIL ET SÉCURISATION DE LA ROTATION
# ============================================================
try:
    print("🚀 Envoi de la super newsletter contenant les 4 produits...")
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as serveur:
        serveur.login(expediteur, mot_de_passe)
        serveur.sendmail(expediteur, destinataire, message.as_string())
    print("🎉 Succès ! L'e-mail regroupant les 4 offres et les 8 pièces jointes graphiques a été envoyé.")
    
    # Sauvegarde du dernier index traité (le 4ème produit de la liste du jour)
    with open(chemin_compteur, 'w') as f:
        f.write(str(indices_aujourdhui[-1]))
    print(f"✅ Rotation mise à jour. Demain, le script repartira automatiquement sur les 4 produits suivants.")

except Exception as e:
    print(f"🚨 Erreur lors de l'envoi : {e}")

------------------------------------------------------------
🛡️ SÉCURITÉ CATALOGUE — VÉRIFICATION DES 4 PRODUITS DU JOUR
------------------------------------------------------------
📦 [1/4] Four métropol multifonction pyrolyse Inox GLEM GFL... | 💰 549€ (Barre: 899€)
📦 [2/4] Cuisinière Alpha 60 x 50 CM GLEM GA650CMIX... | 💰 499€ (Barre: 815€)
📦 [3/4] RÉFRIGÉRATEUR DOUBLE PORTES ENCASTRABLE  AIRLUX AR... | 💰 472€ (Barre: 779€)
📦 [4/4] Four convection émail lisse Inox GLEM GFM410IX... | 💰 270€ (Barre: 443€)
------------------------------------------------------------


👉 Les prix de vente sont-ils corrects et à jour ? (OUI/NON) :  OUI


🤖 IA : Rédaction des textes marketing pour les 4 produits...
🎨 Génération des rendus HD pour le produit 1/4...
🎨 Génération des rendus HD pour le produit 2/4...
🎨 Génération des rendus HD pour le produit 3/4...
🎨 Génération des rendus HD pour le produit 4/4...
🚀 Envoi de la super newsletter contenant les 4 produits...
